# Bronze Layer — API Ingestion via Databricks (All 17 Entities)

**Day 7 | Equivalent of ADF `pl_bronze_api_master_v4` + `pl_bronze_api_ingest_v4`**

Replicates the full ADF v4 metadata-driven pipeline logic in pure Python:
- Reads entity config from the Bronze Volume (`/Volumes/.../bronze-volume/config/`)
- Authenticates to VoltGrid API using credentials from Key Vault (`kv-ev-scope`)
- Per-entity: reads watermark CSV, paginates the API, writes JSON pages to Bronze Volume
- Writes audit row per entity (always), advances watermark only on success
- All 17 entities run via `ThreadPoolExecutor` (parallel — same as ADF ForEach)

**Storage access:** Unity Catalog Bronze Volume — `/Volumes/dbw_ev_intelligence_dev/default/bronze-volume/`
No ADLS credentials needed. UC External Location + Storage Credential handles auth transparently.

---

### ADF → Databricks activity mapping

| ADF Activity | Databricks equivalent |
|---|---|
| `act_get_username / act_get_password` | `dbutils.secrets.get(scope="kv-ev-scope", key=...)` |
| `act_api_login` | `requests.post(AUTH_ENDPOINT, json=payload)` |
| `act_set_token` | Python variable `TOKEN` |
| `act_set_ingestion_date` | `datetime.now(UTC).strftime('%Y-%m-%d')` |
| `act_get_watermark` | `open(BRONZE_VOLUME/audit/watermark_<entity>.csv)` via Volume path |
| `act_set_watermark` | `EPOCH_WATERMARK` if full, else value from CSV |
| `act_get_total_pages` | GET page 1 → read `pagination.total_pages` |
| `act_paginate` (Until loop) | `for page in range(1, total_pages + 1)` |
| `act_copy_entity_page` | `requests.get(url)` → write JSON to Volume path |
| `act_write_audit` | Append CSV row to Volume audit file (always) |
| `act_write_watermark` | Overwrite Volume watermark file (success only) |
| Master ForEach (parallel, batchCount=20) | `ThreadPoolExecutor(max_workers=17)` |

---

### Bronze Volume output structure (identical to ADF Bronze ADLS output)

```
/Volumes/dbw_ev_intelligence_dev/default/bronze-volume/
  ├── config/
  │     └── pipeline_metadata_config.json
  ├── audit/
  │     ├── pipeline_audit.csv                ← append-only, 1 row per entity per run
  │     ├── watermark_payments.csv            ← overwritten on success
  │     └── watermark_<entity>.csv × 17
  └── api/
        └── <entity>/
              └── ingestion_date=<YYYY-MM-DD>/
                    ├── page_1.json
                    └── page_N.json
```

---

### `load_type` parameter

| Value | Watermark sent to API | Effect |
|---|---|---|
| `full` | `1900-01-01T00:00:00Z` (epoch) | API returns all records — full historical load |
| `incremental` | Value from `watermark_<entity>.csv` | API returns only records updated after last run |

## Cell 1 — Imports

Standard library + `requests` (pre-installed on all Databricks Runtime versions). No `%pip install` needed.

In [ ]:
import json
import io
import csv
import uuid
import requests
import concurrent.futures
from datetime import datetime, timezone

print("Imports OK")

## Cell 2 — Read `load_type` widget parameter

Passed by the Databricks Job as a parameter. Default is `incremental`.

| `load_type` | Behaviour |
|---|---|
| `full` | Uses epoch watermark — fetches all records for all entities |
| `incremental` | Uses per-entity watermark CSV — fetches only new/updated records |

In [ ]:
dbutils.widgets.text("load_type", "incremental", "Load Type (full / incremental)")
LOAD_TYPE = dbutils.widgets.get("load_type").strip().lower()

if LOAD_TYPE not in ("full", "incremental"):
    raise ValueError(f"Invalid load_type='{LOAD_TYPE}'. Must be 'full' or 'incremental'.")

print(f"load_type : {LOAD_TYPE}")

## Cell 3 — Configuration constants

All constants mirror what is hardcoded in the ADF pipeline or its linked services.

In [ ]:
# Key Vault secret scope (backed by kv-ev-intelligence-dev)
KV_SCOPE        = "kv-ev-scope"

# VoltGrid API
API_BASE_URL    = "https://ev-project-navy-mu.vercel.app"
AUTH_ENDPOINT   = f"{API_BASE_URL}/api/auth/login/"
EPOCH_WATERMARK = "1900-01-01T00:00:00Z"   # full load — API returns all records

# Unity Catalog Bronze Volume — UC External Location handles auth transparently
# No ADLS credentials needed here
BRONZE_VOLUME   = "/Volumes/dbw_ev_intelligence_dev/default/bronze-volume"

# Subpaths within the volume
CONFIG_PATH     = "config/pipeline_metadata_config.json"
AUDIT_CSV_PATH  = "audit/pipeline_audit.csv"
WATERMARK_DIR   = "audit"
API_DATA_DIR    = "api"

# Audit: pipeline name written to pipeline_audit.csv (matches ADF value)
PIPELINE_NAME   = "pl_bronze_api_ingest_databricks_v1"

print(f"API base       : {API_BASE_URL}")
print(f"Bronze Volume  : {BRONZE_VOLUME}")
print(f"load_type      : {LOAD_TYPE}")

In [ ]:
def volume_read_text(subpath):
    """Read a text file from the Bronze Volume."""
    with open(f"{BRONZE_VOLUME}/{subpath}", "r", encoding="utf-8") as f:
        return f.read()


def volume_write_text(subpath, content: str):
    """Write text to the Bronze Volume, creating parent dirs automatically via dbutils."""
    full_path = f"{BRONZE_VOLUME}/{subpath}"
    dbutils.fs.put(full_path, content, overwrite=True)


def volume_append_csv_row(subpath, row: dict, fieldnames: list):
    """
    Append one CSV row to a file in the Bronze Volume.
    Reads current content, appends the new row, overwrites the file.
    Called sequentially after all threads join — no concurrent write risk.
    """
    full_path = f"{BRONZE_VOLUME}/{subpath}"
    try:
        existing = volume_read_text(subpath)
    except FileNotFoundError:
        existing = ",".join(fieldnames) + "\n"

    buf = io.StringIO()
    writer = csv.DictWriter(buf, fieldnames=fieldnames, quoting=csv.QUOTE_ALL, lineterminator="\n")
    writer.writerow(row)
    new_content = existing.rstrip("\n") + "\n" + buf.getvalue()
    dbutils.fs.put(full_path, new_content, overwrite=True)


def read_watermark(entity_name):
    """Read per-entity watermark value from bronze/audit/watermark_<entity>.csv."""
    subpath = f"{WATERMARK_DIR}/watermark_{entity_name}.csv"
    try:
        content = volume_read_text(subpath)
        reader  = csv.DictReader(io.StringIO(content))
        row     = next(reader)
        return row["watermark_value"]
    except Exception as e:
        print(f"  [{entity_name}] WARNING: Could not read watermark ({e}) — using epoch")
        return EPOCH_WATERMARK


def write_watermark(entity_name, timestamp_utc):
    """Overwrite per-entity watermark CSV with the new UTC timestamp."""
    subpath = f"{WATERMARK_DIR}/watermark_{entity_name}.csv"
    content = (
        "watermark_value,entity_name,updated_at\n"
        f"{timestamp_utc},{entity_name},{timestamp_utc}\n"
    )
    volume_write_text(subpath, content)


print("Helper functions defined — OK")

## Cell 5 — Read entity metadata config from Bronze Volume

ADF equivalent: `act_read_metadata` Lookup activity reading `bronze/config/pipeline_metadata_config.json`.

Returns list of 17 entity dicts with `entity_name`, `api_path`, `page_size`, `enabled`.

In [ ]:
config_json  = volume_read_text(CONFIG_PATH)
all_entities = json.loads(config_json)

# Filter to enabled entities only
entities = [e for e in all_entities if e.get("enabled", True)]

print(f"Entities loaded: {len(entities)}")
for e in entities:
    print(f"  {e['entity_name']:<20}  path={e['api_path']:<30}  page_size={e['page_size']}")

## Cell 6 — VoltGrid API authentication

ADF equivalent: `act_get_username` + `act_get_password` (Key Vault WebActivity) + `act_api_login` (POST WebActivity).

Credentials come from the `kv-ev-scope` secret scope backed by `kv-ev-intelligence-dev`.
One login call is shared across all 17 parallel entity threads.

In [ ]:
# ADF: act_get_username + act_get_password — Key Vault via MSI
# Databricks: Key Vault via secret scope kv-ev-scope
USERNAME = dbutils.secrets.get(scope=KV_SCOPE, key="voltgrid-username")
PASSWORD = dbutils.secrets.get(scope=KV_SCOPE, key="voltgrid-password")

# ADF: act_api_login — POST to /api/auth/login/
resp = requests.post(
    AUTH_ENDPOINT,
    json    = {"username": USERNAME, "password": PASSWORD},
    headers = {"Content-Type": "application/json"},
    timeout = 30
)
resp.raise_for_status()

# ADF: act_set_token — store in variable v_token
TOKEN       = resp.json()["token"]
AUTH_HEADER = {"Authorization": f"Token {TOKEN}"}

print("VoltGrid API authenticated — OK")
print(f"Token : [REDACTED — {len(TOKEN)} chars]")

## Cell 7 — Per-entity ingestion function

Replicates the full ADF child pipeline `pl_bronze_api_ingest_v4` for one entity:

```
read watermark → set watermark (full/incremental) → get total_pages
→ loop pages → write JSON to Volume → collect audit row → write watermark (on success)
```

JSON pages are written directly to the Bronze Volume path:
`/Volumes/.../bronze-volume/api/<entity>/ingestion_date=<date>/page_N.json`

In [ ]:
AUDIT_FIELDNAMES = [
    "pipeline_name", "entity_name", "load_type", "watermark_value",
    "ingestion_date", "total_pages", "status", "pipeline_run_id", "run_timestamp"
]


def ingest_entity(entity: dict, ingestion_date: str, run_id: str) -> dict:
    """
    Full ingestion pipeline for one entity — mirrors pl_bronze_api_ingest_v4.
    Returns dict: {entity_name, status, total_pages, pages_written, watermark, error, audit_row}
    """
    entity_name   = entity["entity_name"]
    api_path      = entity["api_path"]
    page_size     = entity["page_size"]

    status        = "failed"
    total_pages   = 0
    pages_written = 0
    error_msg     = None
    watermark     = EPOCH_WATERMARK

    try:
        # ── act_get_watermark ─────────────────────────────────────────────────
        csv_watermark = read_watermark(entity_name)

        # ── act_set_watermark (full vs incremental) ───────────────────────────
        # ADF: IF p_load_type == 'full' THEN epoch ELSE csv value
        watermark = EPOCH_WATERMARK if LOAD_TYPE == "full" else csv_watermark

        print(f"  [{entity_name}] load={LOAD_TYPE}  watermark={watermark}")

        # ── act_get_total_pages — GET page 1 to read pagination.total_pages ───
        page1_url = (
            f"{API_BASE_URL}{api_path}"
            f"?page=1&page_size={page_size}&updated_after={watermark}"
        )
        r1 = requests.get(page1_url, headers=AUTH_HEADER, timeout=60)
        r1.raise_for_status()
        page1_data  = r1.json()
        total_pages = page1_data["pagination"]["total_pages"]

        print(f"  [{entity_name}] total_pages={total_pages}")

        # ── act_paginate (Until loop) → act_copy_entity_page ─────────────────
        for page in range(1, total_pages + 1):
            page_data = page1_data if page == 1 else None

            if page_data is None:
                url  = (
                    f"{API_BASE_URL}{api_path}"
                    f"?page={page}&page_size={page_size}&updated_after={watermark}"
                )
                resp = requests.get(url, headers=AUTH_HEADER, timeout=60)
                resp.raise_for_status()
                page_data = resp.json()

            # Write JSON page to Bronze Volume
            # Path: /Volumes/.../bronze-volume/api/<entity>/ingestion_date=<date>/page_N.json
            subpath = f"{API_DATA_DIR}/{entity_name}/ingestion_date={ingestion_date}/page_{page}.json"
            volume_write_text(subpath, json.dumps(page_data))
            pages_written += 1

        # ── act_set_status_success ────────────────────────────────────────────
        status = "succeeded"
        print(f"  [{entity_name}] DONE — {pages_written} page(s) written")

    except Exception as e:
        # ── act_set_status_failed ─────────────────────────────────────────────
        status    = "failed"
        error_msg = str(e)
        print(f"  [{entity_name}] FAILED — {error_msg}")

    finally:
        # ── act_write_audit row is built here and written by Cell 8 after threads join ──
        run_timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
        audit_row = {
            "pipeline_name":   PIPELINE_NAME,
            "entity_name":     entity_name,
            "load_type":       LOAD_TYPE,
            "watermark_value": watermark,
            "ingestion_date":  ingestion_date,
            "total_pages":     str(total_pages),
            "status":          status,
            "pipeline_run_id": run_id,
            "run_timestamp":   run_timestamp
        }

    return {
        "entity_name":   entity_name,
        "status":        status,
        "total_pages":   total_pages,
        "pages_written": pages_written,
        "watermark":     watermark,
        "error":         error_msg,
        "audit_row":     audit_row,
        "run_timestamp": audit_row["run_timestamp"]
    }


print("ingest_entity() defined — OK")

## Cell 8 - Run all entities in parallel + write audit & watermarks

ADF equivalent: `act_foreach_entity` ForEach (`isSequential: false`, `batchCount: 20`).

`ThreadPoolExecutor(max_workers=17)` runs all 17 entities concurrently.
After all threads join, audit rows are written sequentially (safe - no concurrent writes to shared file).
Watermarks are advanced only for succeeded entities - same as ADF `act_write_watermark` dependsOn success.

In [ ]:
succeeded = [r for r in results if r["status"] == "succeeded"]
failed    = [r for r in results if r["status"] == "failed"]

print("=" * 70)
print("BRONZE API INGESTION - RUN SUMMARY")
print("=" * 70)
print(f"load_type      : {LOAD_TYPE}")
print(f"ingestion_date : {ingestion_date}")
print(f"run_id         : {run_id}")
print(f"entities total : {len(results)}")
print(f"succeeded      : {len(succeeded)}")
print(f"failed         : {len(failed)}")
print()
print(f"  {'Entity':<22} {'Status':<12} {'Pages':<8} {'Watermark used'}")
print(f"  {'-'*22} {'-'*12} {'-'*8} {'-'*30}")
for r in sorted(results, key=lambda x: x["entity_name"]):
    marker = "OK" if r["status"] == "succeeded" else "FAIL"
    print(f"  [{marker}] {r['entity_name']:<20} {r['status']:<12} {r['pages_written']:<8} {r['watermark']}")
    if r["error"]:
        print(f"        Error: {r['error']}")
print()
print(f"Audit rows written to  : {BRONZE_VOLUME}/{AUDIT_CSV_PATH}")
print(f"Watermarks updated     : {len(succeeded)} entity file(s)")
print("=" * 70)

if failed:
    raise Exception(
        f"{len(failed)} entity(ies) failed: "
        + ", ".join(r["entity_name"] for r in failed)
        + " - check output above for details."
    )

## Cell 9 - Run summary

Prints a per-entity status table. Raises an exception if any entity failed so the Databricks Job marks the run as Failed and triggers an email alert.

In [ ]:
# ADF: act_set_ingestion_date — formatDateTime(utcNow(), 'yyyy-MM-dd')
now            = datetime.now(timezone.utc)
ingestion_date = now.strftime("%Y-%m-%d")
run_id         = str(uuid.uuid4())   # ADF: pipeline().RunId

print(f"Ingestion date : {ingestion_date}")
print(f"Run ID         : {run_id}")
print(f"Entities       : {len(entities)}")
print(f"load_type      : {LOAD_TYPE}")
print("\nStarting parallel ingestion...\n")

results = []

# ADF: ForEach (parallel, batchCount=20)
with concurrent.futures.ThreadPoolExecutor(max_workers=17) as executor:
    futures = {
        executor.submit(ingest_entity, entity, ingestion_date, run_id): entity["entity_name"]
        for entity in entities
    }
    for future in concurrent.futures.as_completed(futures):
        results.append(future.result())

print("\nAll entities complete. Writing audit rows and watermarks...")

# act_write_audit — always, for every entity, sequential after threads join
for r in results:
    try:
        volume_append_csv_row(AUDIT_CSV_PATH, r["audit_row"], AUDIT_FIELDNAMES)
        print(f"  [audit] {r['entity_name']} — written")
    except Exception as e:
        print(f"  [audit] {r['entity_name']} — FAILED: {e}")

# act_write_watermark — success only
wm_timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
for r in results:
    if r["status"] == "succeeded":
        try:
            write_watermark(r["entity_name"], wm_timestamp)
            print(f"  [watermark] {r['entity_name']} -> {wm_timestamp}")
        except Exception as e:
            print(f"  [watermark] {r['entity_name']} — FAILED: {e}")

print("\nAudit and watermark writes complete.")